In [ ]:
# CELL 1 - environment probe and raw-file check (nothing is modified, no model is trained)
# Carnets ships NumPy, pandas and matplotlib; packages with compiled code (SciPy, scikit-learn, lifelines,
# scikit-survival, XGBoost, PyTorch) cannot be installed on iOS, so the toolkit in Cell 6 is NumPy-only.
import sys, platform, importlib, hashlib, glob, os

print("Python", sys.version.split()[0], "|", platform.machine())
for name in ["numpy", "pandas", "matplotlib", "openpyxl", "scipy", "sklearn", "lifelines", "sksurv", "xgboost", "torch"]:
    try:
        m = importlib.import_module(name)
        print(f"  OK       {name:12s} {getattr(m, '__version__', '')}")
    except Exception as e:
        print(f"  MISSING  {name:12s} ({type(e).__name__})")

# the two Dryad files (any spelling of the names, e.g. 'RC+Health+Care+Data-20180820.xlsx'); SHA-256 from the kit
EXPECTED = {"RICH":   ("*RC*Health*Care*Data*.xlsx", "ffd341a1a91ac65585b9fba70847a2681f5671a6035d9cfbf92e50795d74fbf6"),
            "NAGALA": ("*Int*J*Obesity*.xlsx",       "671359aea2019c848f969395f88e57f888eaba8463ea0351bcda8f1c947f5523")}

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

print("\nWorking folder:", os.getcwd())
for cohort, (pattern, sha) in EXPECTED.items():
    hits = sorted(glob.glob(pattern))
    if not hits:
        print(f"  {cohort}: NOT FOUND - copy the .xlsx into this folder (same folder as the notebook)")
        continue
    ok = "checksum OK" if sha256(hits[0]) == sha else "CHECKSUM DIFFERS - not the Dryad file"
    print(f"  {cohort}: {hits[0]}  {os.path.getsize(hits[0]) / 1e6:.1f} MB  {ok}")


In [ ]:
# CELL 2 - install openpyxl (pure Python, installs fine on iOS) and make it importable in this kernel
import sys, site, subprocess, importlib

usp = site.getusersitepackages()                 # pip installs to the user site on Carnets; add it to sys.path
if usp not in sys.path:
    sys.path.append(usp)
try:
    import openpyxl
    print("openpyxl already present:", openpyxl.__version__)
except ModuleNotFoundError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "openpyxl"], capture_output=True, text=True)
    print(r.stdout[-800:], r.stderr[-800:])
    importlib.invalidate_caches()
    import openpyxl
    print("openpyxl installed:", openpyxl.__version__)


In [ ]:
# CELL 3 - verify checksums, read both workbooks once (slow), cache as CSV so every later cell loads in seconds
import sys, site, glob, time, pathlib
import pandas as pd
usp = site.getusersitepackages()
if usp not in sys.path:
    sys.path.append(usp)

RICH_XLSX = sorted(glob.glob("*RC*Health*Care*Data*.xlsx"))[0]
NAGALA_XLSX = sorted(glob.glob("*Int*J*Obesity*.xlsx"))[0]
EXPECTED = {RICH_XLSX: "ffd341a1a91ac65585b9fba70847a2681f5671a6035d9cfbf92e50795d74fbf6",
            NAGALA_XLSX: "671359aea2019c848f969395f88e57f888eaba8463ea0351bcda8f1c947f5523"}
for f, sha in EXPECTED.items():
    assert sha256(f) == sha, f"checksum differs: {f}"
print("checksums OK:", RICH_XLSX, "|", NAGALA_XLSX)

CACHE = pathlib.Path("ckinn_cache"); CACHE.mkdir(exist_ok=True)
for name, f in [("rich_raw", RICH_XLSX), ("nagala_raw", NAGALA_XLSX)]:
    out = CACHE / f"{name}.csv"
    if out.exists():
        print(f"{name}: cached already ({out})"); continue
    t = time.time()
    df = pd.read_excel(f, engine="openpyxl")
    df.to_csv(out, index=False)
    print(f"{name}: read {df.shape[0]:,} rows x {df.shape[1]} columns in {time.time() - t:.0f} s -> {out}")

rich_raw = pd.read_csv(CACHE / "rich_raw.csv")
nag_raw = pd.read_csv(CACHE / "nagala_raw.csv")
assert rich_raw.shape == (211833, 25) and nag_raw.shape == (15464, 31), (rich_raw.shape, nag_raw.shape)
print("\nRICH  ", rich_raw.shape, "| NAGALA", nag_raw.shape)
print("RICH columns:", list(rich_raw.columns), "\n")
print("NAGALA columns:", list(nag_raw.columns))


In [ ]:
# CELL 4 - harmonise RICH and NAGALA into one schema (protocol sections 3-4, data_dictionary.md); missing stays missing
import pathlib
import numpy as np, pandas as pd
CACHE = pathlib.Path("ckinn_cache")
rich_raw = pd.read_csv(CACHE / "rich_raw.csv")              # reload from cache: safe after a kernel restart
nag_raw = pd.read_csv(CACHE / "nagala_raw.csv")

r = rich_raw.copy()
r.columns = ['id','age','sex_raw','site','height','weight','bmi','sbp','dbp','fpg','tc','tg','hdl','ldl',
             'alt','ast','bun','ccr','fpg_final','dm_dx','censor','time_y','smoke_raw','drink_raw','famhx']
dx = r.dm_dx.fillna(0).eq(1)
assert ((r.censor == 1) == (dx | (r.fpg_final >= 7))).all(), "RICH event definition changed"
rich = pd.DataFrame({
    'cohort': 'RICH', 'id': r.id, 'site': r.site, 'age': r.age,
    'sex_male': (r.sex_raw == 1).astype(int),                 # file codes 1 male, 2 female
    'bmi': r.bmi, 'sbp': r.sbp, 'dbp': r.dbp, 'fpg': r.fpg, 'tc': r.tc, 'tg': r.tg, 'hdl': r.hdl,
    'alt': r.alt, 'ast': r.ast,
    'smoke': r.smoke_raw.map({3: 0, 2: 1, 1: 2}),             # 0 never, 1 former, 2 current
    'famhx': r.famhx, 'hba1c': np.nan, 'exercise': np.nan, 'waist': np.nan, 'fatty_liver': np.nan, 'ggt': np.nan,
    'time_y': r.time_y, 'event': r.censor.astype(int), 'event_dx_only': dx.astype(int)})

n = nag_raw.copy()
n.columns = ['id','sex_raw','age','ethanol','fatty','bmi','wc','wc_hi','bmi25','phen','days','alt','ast','wt','exer',
             'ggt','hdl_mg','hdl','tc_mg','tc','tg_mg','tg','a1c','a1c_mm','alc','smoke_raw','fpg_mg','fpg','sbp','dbp','dm']
assert set(n.sex_raw.unique()) <= {0, 1}, "NAGALA sex coding changed"
nag = pd.DataFrame({
    'cohort': 'NAGALA', 'id': n.id, 'site': 1, 'age': n.age,
    'sex_male': n.sex_raw.astype(int),                        # data are 0/1; 1 = male (protocol decision D2)
    'bmi': n.bmi, 'sbp': n.sbp, 'dbp': n.dbp, 'fpg': n.fpg, 'tc': n.tc, 'tg': n.tg, 'hdl': n.hdl,
    'alt': n.alt, 'ast': n.ast,
    'smoke': n.smoke_raw.map({1: 0, 2: 1, 3: 2}),
    'famhx': np.nan, 'hba1c': n.a1c, 'exercise': n.exer, 'waist': n.wc, 'fatty_liver': n.fatty, 'ggt': n.ggt,
    'time_y': n.days / 365.25, 'event': n.dm.astype(int), 'event_dx_only': np.nan})

for d in (rich, nag):
    d['age_band'] = pd.cut(d.age, [0, 40, 55, 200], right=False, labels=['<40', '40-54', '55+'])
rich.to_csv(CACHE / "rich_harmonized.csv", index=False)
nag.to_csv(CACHE / "nagala_harmonized.csv", index=False)

# locked protocol numbers (sections 2-3) and the source-paper sex count (decision D2)
assert len(rich) == 211833 and rich.event.sum() == 4174 and rich.event_dx_only.sum() == 1304
assert len(nag) == 15464 and nag.event.sum() == 373 and nag.sex_male.sum() == 8430
print("Protocol numbers reproduced: RICH 211,833 / 4,174 events / 1,304 diagnosed; NAGALA 15,464 / 373 events; 8,430 NAGALA men")

def pct(x): return f"{100 * x:.1f}%"
rows = [("Participants",            lambda d: f"{len(d):,}"),
        ("Events (rate)",           lambda d: f"{d.event.sum():,} ({pct(d.event.mean())})"),
        ("Follow-up years, mean (min-max)", lambda d: f"{d.time_y.mean():.2f} ({d.time_y.min():.2f}-{d.time_y.max():.2f})"),
        ("Median follow-up, years", lambda d: f"{d.time_y.median():.2f}"),
        ("Incidence per 1,000 person-years", lambda d: f"{1000 * d.event.sum() / d.time_y.sum():.2f}"),
        ("Male",                    lambda d: pct(d.sex_male.mean())),
        ("Age, mean (min-max)",     lambda d: f"{d.age.mean():.1f} ({d.age.min():.0f}-{d.age.max():.0f})"),
        ("BMI, mean",               lambda d: f"{d.bmi.mean():.1f}"),
        ("FPG, mean (max)",         lambda d: f"{d.fpg.mean():.2f} ({d.fpg.max():.2f})")]
profile = pd.DataFrame({lab: {"RICH": f(rich), "NAGALA": f(nag)} for lab, f in rows}).T
print(); print(profile.to_string())

miss = pd.DataFrame({"RICH %": rich.isna().mean() * 100, "NAGALA %": nag.isna().mean() * 100}).round(1)
miss = miss.loc[['tc', 'tg', 'hdl', 'alt', 'ast', 'smoke', 'famhx', 'hba1c', 'exercise', 'waist', 'fatty_liver', 'ggt']]
print("\nMissingness (100 = not collected):"); print(miss.to_string())


In [ ]:
# CELL 5 - do the clinical priors (protocol section 7) hold descriptively? Event rate by quintile; Kaplan-Meier by cohort
import pathlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
CACHE = pathlib.Path("ckinn_cache")
rich = pd.read_csv(CACHE / "rich_harmonized.csv")
nag = pd.read_csv(CACHE / "nagala_harmonized.csv")
FIG = pathlib.Path("ckinn_figs"); FIG.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

H = 3.0                                                        # primary horizon, years (protocol section 6)
PRIORS = {'age': +1, 'bmi': +1, 'sbp': +1, 'tg': +1, 'fpg': +1, 'hdl': -1,
          'famhx': +1, 'hba1c': +1, 'exercise': -1}            # +1 risk rises with the variable, -1 falls

def km(time, event, grid):
    """Kaplan-Meier cumulative incidence on a time grid (pure NumPy)."""
    t = np.asarray(time, float); e = np.asarray(event, int)
    order = np.argsort(t); t, e = t[order], e[order]
    ut, idx = np.unique(t, return_index=True)
    d = np.add.reduceat(e, idx)                                # events at each unique time
    n_at_risk = len(t) - idx                                   # at risk just before each unique time
    surv = np.cumprod(1 - d / n_at_risk)
    pos = np.searchsorted(ut, grid, side="right") - 1
    return np.where(pos >= 0, 1 - surv[np.clip(pos, 0, None)], 0.0)

# 1. event rate by quintile for the shared continuous priors
vars_ = ['age', 'bmi', 'sbp', 'tg', 'fpg', 'hdl']
fig, axes = plt.subplots(2, 6, figsize=(16, 5), sharey='row')
summary = []
for r_, (name, d) in enumerate([("RICH", rich), ("NAGALA", nag)]):
    for c_, v in enumerate(vars_):
        q = pd.qcut(d[v], 5, labels=False, duplicates='drop')
        rate = d.groupby(q).event.mean() * 100
        ax = axes[r_, c_]
        ax.plot(range(len(rate)), rate.values, 'o-', color='#E45756' if PRIORS[v] < 0 else '#4C78A8')
        ax.set_title(f"{name}: {v} (prior {'+' if PRIORS[v] > 0 else '-'})", fontsize=9); ax.set_xlabel("quintile")
        summary.append({"cohort": name, "variable": v, "prior": "+" if PRIORS[v] > 0 else "-",
                        "Q1 %": round(rate.values[0], 2), "Q5 %": round(rate.values[-1], 2),
                        "direction agrees": bool((rate.values[-1] - rate.values[0]) * PRIORS[v] > 0),
                        "monotone across quintiles": bool(np.all(np.diff(rate.values) * PRIORS[v] >= 0))})
axes[0, 0].set_ylabel("event rate, %"); axes[1, 0].set_ylabel("event rate, %")
plt.tight_layout(); plt.savefig(FIG / "05_priors_vs_data.png"); plt.show()
print(pd.DataFrame(summary).to_string(index=False))

# 2. binary priors (cohort-specific extras)
print("\nRICH family history: event rate %", rich.groupby('famhx').event.mean().mul(100).round(2).to_dict())
print("NAGALA exercise habit: event rate %", nag.groupby('exercise').event.mean().mul(100).round(2).to_dict())
q = pd.qcut(nag.hba1c, 5, labels=False, duplicates='drop')
print("NAGALA HbA1c quintiles: event rate %", nag.groupby(q).event.mean().mul(100).round(2).tolist())

# 3. Kaplan-Meier cumulative incidence and the censoring problem at the 3-year horizon
grid = np.linspace(0, 8, 161)
fig, ax = plt.subplots(figsize=(6, 3.8))
for name, d in [("RICH", rich), ("NAGALA", nag)]:
    ax.plot(grid, km(d.time_y, d.event, grid) * 100, label=name)
ax.axvline(H, ls='--', c='grey'); ax.set_xlabel("years"); ax.set_ylabel("cumulative incidence, %"); ax.legend()
plt.tight_layout(); plt.savefig(FIG / "05_km.png"); plt.show()
for name, d in [("RICH", rich), ("NAGALA", nag)]:
    print(f"{name}: KM 3-year incidence {km(d.time_y, d.event, [H])[0] * 100:.2f}% | "
          f"censored before 3 y {((d.event == 0) & (d.time_y < H)).mean():.1%} | "
          f"followed >= 3 y {(d.time_y >= H).mean():.1%}")


In [ ]:
# CELL 6 - write the NumPy-only survival toolkit (ckinn_np.py) next to the notebook, self-test on toy arrays, smoke-test on one real fold
import pathlib, importlib, sys, time
import numpy as np, pandas as pd

SRC = r'''"""ckinn_np: NumPy-only survival toolkit for the CKINN notebooks (runs on Carnets/iPad).

Replaces lifelines, scikit-survival, scikit-learn and SciPy. Every model exposes .fit(df) and .lp(df)
(log relative hazard); `fit_with_baseline` adds a Breslow baseline so each model gives absolute 3-year risk.
Conventions follow the kit's src/ckinn/lib.py so results are comparable with the cloud reference runs.
Real data only; no imputation beyond training-fold medians plus missingness indicators (protocol section 5).
"""
import numpy as np
import pandas as pd

H = 3.0                                   # primary horizon, years (protocol section 6)
BASE = ['age', 'sex_male', 'bmi', 'sbp', 'dbp']
TIERS = {'T1': BASE,
         'T2': BASE + ['tc', 'tg', 'alt'],
         'T3': BASE + ['tc', 'tg', 'alt', 'fpg'],
         'T4': BASE + ['tc', 'tg', 'alt', 'fpg', 'hdl', 'ast']}
PRIORS = {'age': +1, 'bmi': +1, 'sbp': +1, 'tg': +1, 'fpg': +1, 'hdl': -1,
          'famhx': +1, 'hba1c': +1, 'exercise': -1}          # protocol section 7
SMALL_SITES = (0, 16)                     # pooled into training, never a test fold (protocol section 8)


# ------------------------------------------------------------------ preprocessing
class Prep:
    """Training-fold median fill, missingness indicators (columns with NaN in training), z-scoring."""
    def __init__(self, cols):
        self.cols = list(cols)

    def fit(self, df):
        X = df[self.cols]
        self.med = X.median()
        self.ind = [c for c in self.cols if X[c].isna().any()]
        Xf = X.fillna(self.med)
        self.mu, self.sd = Xf.mean(), Xf.std().replace(0, 1)
        self.names = self.cols + [c + '_missing' for c in self.ind]
        return self

    def transform(self, df):
        X = df[self.cols]
        out = ((X.fillna(self.med) - self.mu) / self.sd).to_numpy(np.float64)
        if self.ind:
            out = np.hstack([out, X[self.ind].isna().to_numpy(np.float64)])
        return out


# ------------------------------------------------------------------ Cox partial likelihood (Efron ties)
def _sort_by_time(t):
    """Descending time order so that the risk set of a tie block is every row up to the block's end."""
    return np.argsort(-t, kind='stable')


def cox_nll_grad_hess(beta, X, t, e, penalizer=0.0, pen_scale=None):
    """Negative log partial likelihood with Efron's tie correction, averaged per observation, plus a ridge
    penalty 0.5 * penalizer * sum_j (pen_scale_j * beta_j)^2 (the objective lifelines' CoxPHFitter maximises,
    where pen_scale_j is the SD of column j). Returns value, gradient and Hessian."""
    pen_scale = np.ones(X.shape[1]) if pen_scale is None else np.asarray(pen_scale, float)
    o = _sort_by_time(t)
    X, t, e = X[o], t[o], e[o]
    n, p = X.shape
    lp = X @ beta
    lp = lp - lp.max()
    r = np.exp(lp)
    rX = r[:, None] * X
    rXX = rX[:, :, None] * X[:, None, :]
    # risk-set sums over rows with time >= t_block (descending order: cumulative sums to the end of the tie block)
    S0, S1, S2 = np.cumsum(r), np.cumsum(rX, axis=0), np.cumsum(rXX, axis=0)
    _, first = np.unique(-t, return_index=True)             # -t ascends in this order: block starts
    starts = np.append(first, n)
    ends = starts[1:] - 1                                   # last index of each block
    # per-block sums over the events in the block (Efron)
    ev = (e == 1).astype(float)
    cE0 = np.cumsum(r * ev); cE1 = np.cumsum(rX * ev[:, None], axis=0); cE2 = np.cumsum(rXX * ev[:, None, None], axis=0)
    cD = np.cumsum(ev)
    def block_sum(c):                                       # sum within each block from cumulative sums
        return c[ends] - np.concatenate([np.zeros_like(c[:1]), c[ends[:-1]]])
    d = block_sum(cD)                                       # events per block
    keep = d > 0
    d, s0, s1, s2 = d[keep], S0[ends][keep], S1[ends][keep], S2[ends][keep]
    e0, e1, e2 = block_sum(cE0)[keep], block_sum(cE1)[keep], block_sum(cE2)[keep]
    xsum = block_sum(np.cumsum(X * ev[:, None], axis=0))[keep]
    lpsum = block_sum(np.cumsum(lp * ev))[keep]
    nll, grad, hess = -lpsum.sum(), -xsum.sum(0), np.zeros((p, p))
    for l in range(int(d.max())):
        m = d > l
        f = (l / d[m])[:, None]
        w0 = s0[m] - f[:, 0] * e0[m]
        w1 = s1[m] - f * e1[m]
        w2 = s2[m] - f[:, :, None] * e2[m]
        mu = w1 / w0[:, None]
        nll += np.log(w0).sum()
        grad += mu.sum(0)
        hess += (w2 / w0[:, None, None] - mu[:, :, None] * mu[:, None, :]).sum(0)
    pen = penalizer * pen_scale ** 2
    nll = nll / n + 0.5 * (pen * beta * beta).sum()
    grad = grad / n + pen * beta
    hess = hess / n + np.diag(pen)
    return nll, grad, hess


class CoxLinear:
    """Ridge-penalised Cox regression by Newton-Raphson on standardised inputs (lifelines-equivalent)."""
    def __init__(self, cols, penalizer=0.01, max_iter=50, tol=1e-7):
        self.cols, self.penalizer, self.max_iter, self.tol = list(cols), penalizer, max_iter, tol

    def fit(self, df):
        self.prep = Prep(self.cols).fit(df)
        X = self.prep.transform(df)
        t, e = df.time_y.to_numpy(float), df.event.to_numpy(int)
        beta = np.zeros(X.shape[1])
        scale = X.std(0, ddof=1); scale[scale == 0] = 1.0        # lifelines penalises on its own normalised scale
        nll, g, Hm = cox_nll_grad_hess(beta, X, t, e, self.penalizer, scale)
        for _ in range(self.max_iter):
            step = np.linalg.solve(Hm, g)
            s = 1.0
            while True:                                     # backtracking line search
                nb = beta - s * step
                nll2, g2, H2 = cox_nll_grad_hess(nb, X, t, e, self.penalizer, scale)
                if nll2 <= nll + 1e-12 or s < 1e-4:
                    break
                s *= 0.5
            conv = np.abs(nb - beta).max() < self.tol
            beta, nll, g, Hm = nb, nll2, g2, H2
            if conv:
                break
        self.beta, self.nll_ = beta, nll
        return self

    def lp(self, df):
        return self.prep.transform(df) @ self.beta


def cox_univariate_slope(z, t, e):
    """Slope of a one-variable unpenalised Cox fit (used for calibration slope and recalibration)."""
    d = pd.DataFrame({'z': np.asarray(z, float), 'time_y': t, 'event': e})
    m = CoxLinear(['z'], penalizer=0.0).fit(d)              # Prep standardises z internally
    return float(m.beta[0] / m.prep.sd['z'])                # back to the original scale of z


# ------------------------------------------------------------------ Kaplan-Meier, Breslow, absolute risk
def km_survival(time, event, at):
    """Kaplan-Meier survival S(at) for a scalar or array of times."""
    t = np.asarray(time, float); e = np.asarray(event, int)
    o = np.argsort(t); t, e = t[o], e[o]
    ut, idx = np.unique(t, return_index=True)
    d = np.add.reduceat(e, idx)
    n = len(t) - idx
    surv = np.cumprod(1 - d / n)
    pos = np.searchsorted(ut, np.atleast_1d(at), side='right') - 1
    out = np.where(pos >= 0, surv[np.clip(pos, 0, None)], 1.0)
    return out if np.ndim(at) else float(out[0])


def km_risk(df, h=H):
    if len(df) < 20 or df.time_y.max() < h or df.event.sum() == 0:
        return np.nan
    return 1 - km_survival(df.time_y, df.event, h)


class Breslow:
    def __init__(self, time, event, lp):
        o = np.argsort(time)
        t, e, r = np.asarray(time)[o], np.asarray(event)[o], np.exp(np.clip(np.asarray(lp)[o], -20, 20))
        S = np.cumsum(r[::-1])[::-1]
        ut, cnt = np.unique(t[e == 1], return_counts=True)
        self.t, self.H = ut, np.cumsum(cnt / S[np.searchsorted(t, ut, side='left')])

    def __call__(self, h):
        i = np.searchsorted(self.t, h, side='right') - 1
        return 0.0 if i < 0 else float(self.H[i])


class Fitted:
    def __init__(self, model, train):
        self.model, self.train = model, train
        self.H0 = Breslow(train.time_y.values, train.event.values, model.lp(train))

    def risk(self, df, h=H):
        lp = np.asarray(self.model.lp(df), dtype=np.float64)
        return 1 - np.exp(-self.H0(h) * np.exp(np.clip(lp, -20, 20)))


def fit_with_baseline(model, train):
    return Fitted(model.fit(train), train)


class ScaledLP:
    """Wraps a model and multiplies its lp by a slope (used for recalibration)."""
    def __init__(self, model, slope):
        self.model, self.slope, self.cols = model, slope, model.cols

    def lp(self, df):
        return self.slope * self.model.lp(df)


# ------------------------------------------------------------------ metrics
class _BIT:
    def __init__(self, n):
        self.n, self.a = n, np.zeros(n + 1)

    def add(self, i):
        i += 1
        while i <= self.n:
            self.a[i] += 1; i += i & -i

    def prefix(self, i):                                      # count of inserted ranks <= i
        i += 1; s = 0.0
        while i > 0:
            s += self.a[i]; i -= i & -i
        return s


def concordance_index(time, lp, event):
    """Harrell's C with censoring: higher lp must mean shorter time. Ties in lp count 0.5 (lifelines convention)."""
    t = np.asarray(time, float); s = np.asarray(lp, float); e = np.asarray(event, int)
    rank = np.unique(s, return_inverse=True)[1]
    o = np.argsort(-t, kind='stable')                         # descending time; process later times first
    t, rank, e = t[o], rank[o], e[o]
    bit, conc, disc, ties, i, n = _BIT(rank.max() + 1), 0.0, 0.0, 0.0, 0, len(t)
    while i < n:
        j = i
        while j < n and t[j] == t[i]:
            j += 1
        for k in range(i, j):                                 # rows with strictly longer time are already in the tree
            if e[k] == 1:
                below = bit.prefix(rank[k] - 1) if rank[k] > 0 else 0.0
                eq = bit.prefix(rank[k]) - below
                total = bit.prefix(bit.n - 1)
                conc += below; ties += eq; disc += total - below - eq
        for k in range(i, j):
            bit.add(rank[k])
        i = j
    return float((conc + 0.5 * ties) / max(conc + disc + ties, 1))


def ipcw_brier(train, test, risk, h=H):
    """IPCW Brier score at h (Graf et al.), censoring distribution from the training data (as in scikit-survival)."""
    t = test.time_y.to_numpy(float); e = test.event.to_numpy(int); S = 1 - np.asarray(risk, float)
    G_t = km_survival(train.time_y, 1 - train.event, t)      # censoring survival at T_i
    G_h = km_survival(train.time_y, 1 - train.event, h)
    case = (t <= h) & (e == 1)
    ctrl = t > h
    out = np.zeros(len(t))
    out[case] = S[case] ** 2 / G_t[case]
    out[ctrl] = (1 - S[ctrl]) ** 2 / G_h
    return float(out.mean())


def cumulative_dynamic_auc(train, test, lp, h=H):
    """IPCW cumulative/dynamic AUC at h (Uno et al.), as in scikit-survival: cases weighted by 1/G(T_i)."""
    t = test.time_y.to_numpy(float); e = test.event.to_numpy(int); s = np.asarray(lp, float)
    case = (t <= h) & (e == 1); ctrl = t > h
    w = 1.0 / km_survival(train.time_y, 1 - train.event, t[case])
    sc, sk = s[case], np.sort(s[ctrl])
    n_le = np.searchsorted(sk, sc, side='right')              # controls with lp <= case lp
    return float((w * n_le).sum() / (w.sum() * len(sk)))


def cal_slope(risk, df):
    z = np.log(-np.log1p(-np.clip(np.asarray(risk, dtype=np.float64), 1e-9, 1 - 1e-9)))
    return cox_univariate_slope(z, df.time_y.to_numpy(float), df.event.to_numpy(int))


def violation_rate(model, df, delta=0.25, n=5000, seed=0):
    """Share of patients where raising a constrained variable by delta SD moves lp against the prior."""
    s = df.sample(min(n, len(df)), random_state=seed).reset_index(drop=True)
    base, per = model.lp(s), {}
    for c in model.cols:
        if c in PRIORS and s[c].nunique() > 2:
            p = s.copy()
            p[c] = p[c] + delta * df[c].std()
            per[c] = float(np.mean(PRIORS[c] * (model.lp(p) - base) < -1e-6))
    return (float(np.mean(list(per.values()))) if per else np.nan), per


def metrics_from_arrays(train, test, lp, risk, h=H, viol=np.nan):
    keep = (test.time_y < train.time_y.max()).to_numpy()
    test, lp, risk = test[keep], np.asarray(lp)[keep], np.asarray(risk, dtype=np.float64)[keep]
    bs = ipcw_brier(train, test, risk, h)
    s0 = km_survival(train.time_y, train.event, h)
    bs0 = ipcw_brier(train, test, np.full(len(test), 1 - s0), h)
    obs = km_risk(test, h)
    slope = cal_slope(risk, test)
    return {'n': len(test), 'events': int(test.event.sum()),
            'cindex': concordance_index(test.time_y, lp, test.event),
            'auc_3y': cumulative_dynamic_auc(train, test, lp, h),
            'brier': bs, 'scaled_brier': 1 - bs / bs0,
            'obs_risk': obs, 'mean_pred': float(risk.mean()), 'oe': float(obs / risk.mean()),
            'cal_slope': slope, 'cal_slope_dist': abs(slope - 1.0), 'viol_rate': viol}


def evaluate(fit, test, h=H):
    test = test[test.time_y < fit.train.time_y.max()]
    return metrics_from_arrays(fit.train, test, fit.model.lp(test), fit.risk(test, h), h,
                               viol=violation_rate(fit.model, test)[0])


def calibration_table_risk(risk, df, h=H, q=10):
    risk = np.asarray(risk, dtype=np.float64)
    g = pd.qcut(risk, q, labels=False, duplicates='drop')
    rows = []
    for i, d in df.reset_index(drop=True).groupby(g):
        rows.append({'decile': int(i) + 1, 'n': len(d), 'pred': risk[d.index].mean(), 'obs': km_risk(d, h)})
    return pd.DataFrame(rows)


def decision_curve_risk(risk, df, thresholds=np.arange(0.01, 0.101, 0.01), h=H):
    risk = np.asarray(risk, dtype=np.float64)
    prev = km_risk(df, h)
    out = []
    for th in thresholds:
        m = risk >= th
        p = m.mean()
        obs = km_risk(df[m], h) if m.sum() >= 20 else np.nan
        nb = p * obs - p * (1 - obs) * th / (1 - th) if not np.isnan(obs) else np.nan
        out.append({'threshold': float(th), 'net_benefit': nb, 'treat_all': prev - (1 - prev) * th / (1 - th)})
    return pd.DataFrame(out)


def subgroup_metrics(fit, test, by, h=H):
    rows = []
    for g, d in test.groupby(by, observed=True):
        if d.event.sum() >= 20 and d.time_y.max() >= h:
            r = evaluate(fit, d, h)
            rows.append({by: g, **{k: r[k] for k in ['n', 'events', 'cindex', 'auc_3y', 'scaled_brier', 'oe', 'cal_slope']}})
    return pd.DataFrame(rows)


# ------------------------------------------------------------------ splits, transport, inference
def cv_folds(df, k=5, seed=0):
    """Stratified k-fold on `event` (shuffled), equivalent to sklearn StratifiedKFold in spirit."""
    rng = np.random.default_rng(seed)
    fold = np.empty(len(df), int)
    for cls in (0, 1):
        idx = np.flatnonzero(df.event.to_numpy() == cls)
        rng.shuffle(idx)
        fold[idx] = np.arange(len(idx)) % k
    for f in range(k):
        yield df.iloc[fold != f], df.iloc[fold == f]


def loso_folds(rich):
    for s in sorted(rich.site.unique()):
        if s in SMALL_SITES:
            continue
        yield s, rich[rich.site != s], rich[rich.site == s]


def shift_split(df, var, cut):
    return df[df[var] < cut], df[df[var] >= cut]


def recalibrate(fit, target_sample):
    """Intercept-and-slope recalibration on a small target sample (new slope, new Breslow baseline)."""
    lp = fit.model.lp(target_sample)
    slope = cox_univariate_slope(lp, target_sample.time_y.to_numpy(float), target_sample.event.to_numpy(int))
    return Fitted(ScaledLP(fit.model, slope), target_sample), slope


def boot_diff_arrays(train, test, arrs, ref, n_boot=200, seed=0, h=H, max_n=None,
                     metrics=('cindex', 'scaled_brier', 'cal_slope_dist')):
    """Paired bootstrap of metric(model) - metric(ref) on the same resampled patients."""
    rng = np.random.default_rng(seed)
    if max_n and len(test) > max_n:
        sel = rng.choice(len(test), max_n, replace=False)
        test, arrs = test.iloc[sel], {k: (v[0][sel], v[1][sel]) for k, v in arrs.items()}
    out = {k: {m: [] for m in metrics} for k in arrs if k != ref}
    for _ in range(n_boot):
        idx = rng.integers(0, len(test), len(test))
        b = test.iloc[idx]
        r_ref = metrics_from_arrays(train, b, arrs[ref][0][idx], arrs[ref][1][idx], h)
        for k in out:
            r_k = metrics_from_arrays(train, b, arrs[k][0][idx], arrs[k][1][idx], h)
            for m in metrics:
                out[k][m].append(r_k[m] - r_ref[m])
    rows = []
    for k, d in out.items():
        for m, v in d.items():
            rows.append({'model': k, 'ref': ref, 'metric': m, 'mean': float(np.mean(v)),
                         'lo95': float(np.percentile(v, 2.5)), 'hi95': float(np.percentile(v, 97.5)), 'n_boot': n_boot})
    return pd.DataFrame(rows)


# two-sided 97.5% Student t quantiles for df 1..30 (no SciPy on the iPad); normal beyond 30
_T975 = [12.706, 4.303, 3.182, 2.776, 2.571, 2.447, 2.365, 2.306, 2.262, 2.228, 2.201, 2.179, 2.160, 2.145, 2.131,
         2.120, 2.110, 2.101, 2.093, 2.086, 2.080, 2.074, 2.069, 2.064, 2.060, 2.056, 2.052, 2.048, 2.045, 2.042]


def nadeau_bengio(diffs, n_train, n_test):
    """Corrected resampled t interval (95%) for the mean of paired fold differences (Nadeau & Bengio 2003)."""
    d = np.asarray(diffs, dtype=float)
    J = len(d)
    se = np.sqrt((1.0 / J + n_test / n_train) * d.var(ddof=1))
    tq = _T975[J - 2] if J - 1 <= 30 else 1.96
    m = d.mean()
    return {'mean': m, 'lo95': m - tq * se, 'hi95': m + tq * se, 'J': J}
'''
pathlib.Path("ckinn_np.py").write_text(SRC)
if "." not in sys.path:
    sys.path.insert(0, ".")
import ckinn_np as N
importlib.reload(N)
print("ckinn_np written and imported:", len(SRC.splitlines()), "lines")

# --- self-tests on hand-made toy arrays only (mirror tests/test_lib.py in the kit; no study data, no results) ---
def toy(n=400, seed=0):
    r = np.random.default_rng(seed)
    x = r.normal(size=n)
    t = r.exponential(8 / np.exp(0.8 * x))
    c = r.uniform(1, 6, n)
    return pd.DataFrame({'age': x, 'bmi': r.normal(size=n), 'time_y': np.minimum(t, c), 'event': (t <= c).astype(int),
                         'sex_male': r.integers(0, 2, n), 'sbp': r.normal(size=n), 'dbp': r.normal(size=n), 'site': r.integers(1, 5, n)})
d = toy()
b = N.Breslow(d.time_y.values, d.event.values, np.zeros(len(d)))
assert b(1) <= b(3) <= b(6)
f = N.fit_with_baseline(N.CoxLinear(['age', 'bmi']), d)
r = f.risk(d)
assert ((r >= 0) & (r <= 1)).all() and np.corrcoef(r, d.age)[0, 1] > 0.5
class M:
    cols = ['age', 'bmi']
    def lp(self, df): return 1.0 * df.age.values + 0.5 * df.bmi.values
class Neg(M):
    def lp(self, df): return -df.age.values
assert N.violation_rate(M(), d)[0] == 0.0 and N.violation_rate(Neg(), d)[1]['age'] == 1.0
for tr_, te_ in N.cv_folds(d, 5, 0):
    assert not set(tr_.index) & set(te_.index)
dd = np.array([0.01, 0.02, 0.015, 0.0, 0.02, 0.01, 0.012, 0.018, 0.005, 0.02, 0.011, 0.014, 0.016, 0.009, 0.013])
nb = N.nadeau_bengio(dd, 1000, 250)
assert nb['hi95'] - nb['lo95'] > 2 * 2.145 * dd.std(ddof=1) / np.sqrt(len(dd))
d.loc[:5, 'site'] = 0
assert 0 not in [s for s, _, _ in N.loso_folds(d)]
# concordance against a brute-force pairwise count
tt, ss, ee = d.time_y.values[:150], d.age.values[:150], d.event.values[:150]
conc = disc = 0
for i in range(150):
    for j in range(150):
        if ee[i] == 1 and tt[i] < tt[j]:
            conc += ss[i] > ss[j]; disc += ss[i] < ss[j]
assert abs(N.concordance_index(tt, ss, ee) - conc / (conc + disc)) < 1e-12
print("toy self-tests passed")

# --- smoke test on real data: Cox T1 on fold 0 of RICH (same fold definition as later cells) ---
CACHE = pathlib.Path("ckinn_cache")
rich = pd.read_csv(CACHE / "rich_harmonized.csv")
tr, te = next(N.cv_folds(rich, 5, 0))
t0 = time.time()
fit = N.fit_with_baseline(N.CoxLinear(N.TIERS['T1']), tr)
t_fit = time.time() - t0
t0 = time.time()
m = N.evaluate(fit, te)
t_eval = time.time() - t0
print(f"\nCox T1, fold 0: train {len(tr):,} / test {len(te):,} ({m['events']} test events) | fit {t_fit:.1f} s, evaluate {t_eval:.1f} s")
print(pd.Series(fit.model.beta, index=fit.model.prep.names).round(4).to_string())
print({k: round(v, 4) for k, v in m.items() if k not in ('n', 'events')})


In [ ]:
# CELL 7 - R1 internal validation, linear Cox baseline: repeated stratified 5-fold CV in RICH, 3 repeats, tiers T1-T4
# Protocol sections 5, 8, 12: all rows used (median fill + missingness indicators inside the model); CV seed = repeat index.
# Resumable: each (tier, repeat) writes ckinn_cache/parts/cv_Cox_<tier>_r<rep>.csv and is skipped if present,
# so the cell can be re-run after an interruption and only the missing units are computed.
import pathlib, sys, time
import numpy as np, pandas as pd
if "." not in sys.path:
    sys.path.insert(0, ".")
import ckinn_np as N

CACHE = pathlib.Path("ckinn_cache"); PARTS = CACHE / "parts"; PARTS.mkdir(exist_ok=True)
rich = pd.read_csv(CACHE / "rich_harmonized.csv")
FOLDS, REPEATS, TIERS = 5, 3, ['T1', 'T2', 'T3', 'T4']

t_all = time.time()
for tier in TIERS:
    cols = N.TIERS[tier]
    for rep in range(REPEATS):
        out = PARTS / f"cv_Cox_{tier}_r{rep}.csv"
        if out.exists():
            print(f"skip {tier} r{rep} (done)"); continue
        t0, rows = time.time(), []
        for k, (tr, te) in enumerate(N.cv_folds(rich, FOLDS, rep)):
            fit = N.fit_with_baseline(N.CoxLinear(cols), tr)
            m = N.evaluate(fit, te)
            rows.append({'model': 'Cox', 'tier': tier, 'repeat': rep, 'fold': k, 'n_train': len(tr), 'n_test': len(te), **m})
        pd.DataFrame(rows).to_csv(out, index=False)
        print(f"done {tier} r{rep} in {time.time() - t0:.0f} s")
print(f"total {time.time() - t_all:.0f} s")

res = pd.concat([pd.read_csv(p) for p in sorted(PARTS.glob("cv_Cox_*_r*.csv"))], ignore_index=True)
res.to_csv(CACHE / "r1_cox.csv", index=False)
show = ['cindex', 'auc_3y', 'scaled_brier', 'oe', 'cal_slope', 'viol_rate']
summary = res.groupby('tier')[show].agg(['mean', 'std']).round(4)
print("\nR1 Cox baseline, 15 folds per tier (mean, SD):"); print(summary.to_string())


In [ ]:
# CELL 8 - neural models in NumPy: MLP (Cox partial likelihood) and CKINN (MLP + soft clinical-constraint penalty), protocol section 6
import pathlib, importlib, sys, time
import numpy as np, pandas as pd
if '.' not in sys.path:
    sys.path.insert(0, '.')
import ckinn_np as N

SRC = r'''"""ckinn_nn: NumPy implementation of the MLP (Cox partial likelihood) and CKINN (MLP + soft clinical-constraint
penalty) models of protocol section 6, with manual backpropagation. Depends only on ckinn_np.

Network: Linear(p,64) ReLU Dropout(0.1) Linear(64,64) ReLU Dropout(0.1) Linear(64,1).
Loss per batch: L_Cox + lambda * sum_j mean_x relu(-s_j * d lp/d x_j)^2, derivatives w.r.t. standardised inputs.
Training: AdamW (lr 2e-3, weight decay 1e-4), batch 8192, up to 40 epochs, early stopping (patience 4) on a 10%
validation split, best weights restored. Same settings as src/ckinn/lib.py (PyTorch) in the kit.
"""
import numpy as np
import ckinn_np as N


def _cox_batch_loss_grad(lp, t, e):
    """Cox negative log partial likelihood of one batch (Breslow ties, sorted inside) and its gradient w.r.t. lp."""
    o = np.argsort(-t, kind='stable')
    lp_o, e_o = lp[o], e[o]
    m = lp_o.max()
    S = np.cumsum(np.exp(lp_o - m))                           # risk-set sums in descending-time order
    lcse = np.log(S) + m
    ne = max(e_o.sum(), 1.0)
    loss = -((lp_o - lcse) * e_o).sum() / ne
    w = np.cumsum((e_o / S)[::-1])[::-1]                      # sum_{i >= k} e_i / S_i
    g_o = (-e_o + np.exp(lp_o - m) * w) / ne
    g = np.empty_like(g_o); g[o] = g_o
    return loss, g


class DeepCox:
    """MLP with Cox partial-likelihood loss; lam > 0 adds the clinical-constraint penalty (= CKINN)."""
    def __init__(self, cols, hidden=(64, 64), lam=0.0, priors=None, epochs=40, batch=8192, lr=2e-3,
                 wd=1e-4, dropout=0.1, seed=0, patience=4, val_frac=0.1, pen_rows=2048, verbose=False):
        assert len(hidden) == 2, "two hidden layers (protocol section 6)"
        self.cols, self.hidden, self.lam = list(cols), hidden, lam
        self.priors = N.PRIORS if priors is None else priors
        self.epochs, self.batch, self.lr, self.wd, self.dropout = epochs, batch, lr, wd, dropout
        self.seed, self.patience, self.val_frac, self.pen_rows, self.verbose = seed, patience, val_frac, pen_rows, verbose

    # ---- forward / backward
    def _init(self, p, rng):
        h1, h2 = self.hidden
        def lin(i, o):                                        # PyTorch default: U(-1/sqrt(fan_in), 1/sqrt(fan_in))
            b = 1.0 / np.sqrt(i)
            return rng.uniform(-b, b, (i, o)), rng.uniform(-b, b, o)
        self.W1, self.b1 = lin(p, h1)
        self.W2, self.b2 = lin(h1, h2)
        self.W3, self.b3 = lin(h2, 1)
        self.W3 = self.W3[:, 0]; self.b3 = float(self.b3[0])

    def _forward(self, X, train, rng):
        z1 = X @ self.W1 + self.b1
        d1 = (z1 > 0).astype(np.float64)
        if train and self.dropout > 0:
            d1 *= rng.random(z1.shape) >= self.dropout
            d1 /= (1 - self.dropout)
        a1 = z1 * d1
        z2 = a1 @ self.W2 + self.b2
        d2 = (z2 > 0).astype(np.float64)
        if train and self.dropout > 0:
            d2 *= rng.random(z2.shape) >= self.dropout
            d2 /= (1 - self.dropout)
        a2 = z2 * d2
        lp = a2 @ self.W3 + self.b3
        return lp, (X, d1, a1, d2, a2)

    def _backward_lp(self, g_lp, cache):
        """Gradients of sum_i g_lp_i * lp_i w.r.t. all parameters."""
        X, d1, a1, d2, a2 = cache
        gW3 = a2.T @ g_lp; gb3 = g_lp.sum()
        ga2 = np.outer(g_lp, self.W3) * d2
        gW2 = a1.T @ ga2; gb2 = ga2.sum(0)
        ga1 = (ga2 @ self.W2.T) * d1
        gW1 = X.T @ ga1; gb1 = ga1.sum(0)
        return gW1, gb1, gW2, gb2, gW3, gb3

    def _penalty(self, cache):
        """Constraint penalty and its gradient w.r.t. weights. With ReLU masks D1, D2 fixed at the current point,
        d lp / d x = W1 (D1 * (W2 (D2 * W3))) is linear in each weight matrix, so the gradient is exact."""
        X, d1, _, d2, _ = cache
        n = X.shape[0]
        v = d2 * self.W3                                      # (n, h2)
        q = v @ self.W2.T                                     # (n, h1)
        r = d1 * q
        g = r @ self.W1.T                                     # (n, p): d lp / d x
        A = np.zeros_like(g)
        pen = 0.0
        for j, s in self.cons:
            u = np.maximum(0.0, -s * g[:, j])
            pen += (u ** 2).mean()
            A[:, j] = -2.0 * s * u / n                        # d pen / d g_j
        gW1 = r.T @ A                                         # (h1, p)^T -> W1 is (p, h1): transpose below
        b = A @ self.W1                                       # (n, h1): d pen / d r
        gq = d1 * b
        gW2 = (v.T @ gq).T                                    # W2 is (h1, h2): dq_k/dW2[m,k] = v_m
        gv = gq @ self.W2                                     # (n, h2)
        gW3 = (d2 * gv).sum(0)
        return pen, gW1.T, gW2, gW3

    # ---- training
    def fit(self, df):
        rng = np.random.default_rng(self.seed)
        self.prep = N.Prep(self.cols).fit(df)
        X = self.prep.transform(df)
        T = df.time_y.to_numpy(float); E = df.event.to_numpy(float)
        self.cons = [(self.prep.names.index(c), float(s)) for c, s in self.priors.items() if c in self.prep.names]
        self._init(X.shape[1], rng)
        params = ['W1', 'b1', 'W2', 'b2', 'W3', 'b3']
        m = {k: np.zeros_like(getattr(self, k)) for k in params}
        v = {k: np.zeros_like(getattr(self, k)) for k in params}
        b1, b2, eps, step = 0.9, 0.999, 1e-8, 0
        perm = rng.permutation(len(X)); nv = int(len(X) * self.val_frac)
        va, tr = perm[:nv], perm[nv:]
        best, best_state, bad = np.inf, None, 0
        self.history_ = []
        for ep in range(self.epochs):
            order = tr[rng.permutation(len(tr))]
            for s0 in range(0, len(order), self.batch):
                b = order[s0:s0 + self.batch]
                if E[b].sum() < 1:
                    continue
                lp, cache = self._forward(X[b], True, rng)
                loss, g_lp = _cox_batch_loss_grad(lp, T[b], E[b])
                grads = dict(zip(params, self._backward_lp(g_lp, cache)))
                if self.lam > 0 and self.cons:
                    pc = tuple(c[:self.pen_rows] for c in cache)
                    pen, pW1, pW2, pW3 = self._penalty(pc)
                    grads['W1'] = grads['W1'] + self.lam * pW1
                    grads['W2'] = grads['W2'] + self.lam * pW2
                    grads['W3'] = grads['W3'] + self.lam * pW3
                step += 1
                for k in params:                              # AdamW (decoupled weight decay, as torch.optim.AdamW)
                    p = getattr(self, k)
                    m[k] = b1 * m[k] + (1 - b1) * grads[k]
                    v[k] = b2 * v[k] + (1 - b2) * grads[k] ** 2
                    mh = m[k] / (1 - b1 ** step); vh = v[k] / (1 - b2 ** step)
                    p = p * (1 - self.lr * self.wd) - self.lr * mh / (np.sqrt(vh) + eps)
                    setattr(self, k, p)
            lp_va, _ = self._forward(X[va], False, rng)
            vloss, _ = _cox_batch_loss_grad(lp_va, T[va], E[va])
            self.history_.append(float(vloss))
            if self.verbose:
                print(f"epoch {ep:02d} val nll {vloss:.4f}")
            if vloss < best - 1e-4:
                best, bad = vloss, 0
                best_state = {k: getattr(self, k).copy() for k in params}
            else:
                bad += 1
                if bad >= self.patience:
                    break
        for k, val in best_state.items():
            setattr(self, k, val)
        self.best_val_nll_, self.epochs_run_ = float(best), len(self.history_)
        return self

    def lp(self, df):
        return self._forward(self.prep.transform(df), False, None)[0]


class CKINNAuto:
    """CKINN with the constraint strength chosen INSIDE the training data (protocol section 6).

    Fit DeepCox on an 80% inner split for lambda in {0} + grid, score scaled Brier on the inner 20%.
    Rule: the largest positive lambda whose inner scaled Brier is within `tol` of the best over all four;
    if none qualifies, the smallest positive lambda (0.1)."""
    def __init__(self, cols, grid=(0.1, 1.0, 10.0), tol=0.002, epochs=40, seed=0, priors=None, verbose=False):
        self.cols, self.grid, self.tol, self.epochs, self.seed, self.priors, self.verbose = list(cols), grid, tol, epochs, seed, priors, verbose

    def fit(self, df):
        fit_part, val = next(N.cv_folds(df, 5, self.seed + 1000))
        sb = {}
        for lam in (0.0,) + tuple(self.grid):
            f = N.fit_with_baseline(DeepCox(self.cols, lam=lam, priors=self.priors, epochs=self.epochs, seed=self.seed), fit_part)
            sb[lam] = N.evaluate(f, val)['scaled_brier']
            if self.verbose:
                print(f"  inner lambda={lam}: scaled Brier {sb[lam]:.4f}")
        best = max(sb.values())
        ok = [l for l in self.grid if sb[l] >= best - self.tol]
        self.lam_ = max(ok) if ok else min(self.grid)
        self.inner_scaled_brier_ = sb
        self.model = DeepCox(self.cols, lam=self.lam_, priors=self.priors, epochs=self.epochs, seed=self.seed).fit(df)
        return self

    def lp(self, df):
        return self.model.lp(df)
'''
pathlib.Path("ckinn_nn.py").write_text(SRC)
import ckinn_nn as NN
importlib.reload(NN)
print("ckinn_nn written and imported:", len(SRC.splitlines()), "lines")

# --- self-test on toy arrays: analytic gradients (Cox loss + penalty, manual backprop) against central differences ---
rng = np.random.default_rng(1)
n, p = 300, 5
X = rng.normal(size=(n, p)); T = rng.exponential(5, n); E = (rng.random(n) < 0.4).astype(float)
m = NN.DeepCox(list('abcde'), lam=1.0, dropout=0.0, seed=0)
m.cons = [(0, 1.0), (1, -1.0), (3, 1.0)]; m._init(p, rng)
def total(model):
    lp, cache = model._forward(X, False, None)
    return NN._cox_batch_loss_grad(lp, T, E)[0] + model.lam * model._penalty(cache)[0]
lp, cache = m._forward(X, False, None)
_, g_lp = NN._cox_batch_loss_grad(lp, T, E)
an = dict(zip(['W1', 'b1', 'W2', 'b2', 'W3', 'b3'], m._backward_lp(g_lp, cache)))
pen, pW1, pW2, pW3 = m._penalty(cache)
an['W1'] = an['W1'] + m.lam * pW1; an['W2'] = an['W2'] + m.lam * pW2; an['W3'] = an['W3'] + m.lam * pW3
worst = 0.0
for k in ['W1', 'b1', 'W2', 'b2', 'W3', 'b3']:
    flat = np.atleast_1d(getattr(m, k)).reshape(-1); a_flat = np.atleast_1d(an[k]).reshape(-1)
    for idx in rng.choice(flat.size, min(15, flat.size), replace=False):
        old = flat[idx]; h = 1e-6
        flat[idx] = old + h; fp = total(m); flat[idx] = old - h; fm = total(m); flat[idx] = old
        worst = max(worst, abs((fp - fm) / (2 * h) - a_flat[idx]))
assert worst < 1e-6, worst
print(f"gradient check passed (max abs error {worst:.1e}); penalty on toy batch {pen:.5f}")

# --- real data, fold 0 of RICH, tier T2: Cox vs MLP vs CKINN at fixed lambda (lambda selection comes in Cell 9) ---
CACHE = pathlib.Path("ckinn_cache")
rich = pd.read_csv(CACHE / "rich_harmonized.csv")
tr, te = next(N.cv_folds(rich, 5, 0))
cols = N.TIERS['T2']
rows = []
for name, make in [('Cox', lambda: N.CoxLinear(cols)),
                   ('MLP', lambda: NN.DeepCox(cols, lam=0.0, seed=0)),
                   ('CKINN lam=1', lambda: NN.DeepCox(cols, lam=1.0, seed=0)),
                   ('CKINN lam=10', lambda: NN.DeepCox(cols, lam=10.0, seed=0))]:
    t0 = time.time(); f = N.fit_with_baseline(make(), tr); t_fit = time.time() - t0
    r = N.evaluate(f, te)
    per = N.violation_rate(f.model, te)[1]
    rows.append({'model': name, 'fit s': round(t_fit), 'epochs': getattr(f.model, 'epochs_run_', ''),
                 **{k: round(r[k], 4) for k in ['cindex', 'auc_3y', 'scaled_brier', 'oe', 'cal_slope', 'viol_rate']},
                 **{'viol ' + k: round(v, 3) for k, v in per.items()}})
    print(f"{name:13s} done in {t_fit:.0f} s")
print(); print(pd.DataFrame(rows).to_string(index=False))


In [ ]:
# CELL 9 - R1 internal validation for the neural models: MLP (unconstrained) and CKINN (lambda chosen inside each
# training fold), repeated stratified 5-fold CV in RICH, 3 repeats. Headline tier T2 first, then T1, T3, T4.
# Protocol section 12 seeds: CV seed = repeat, model seed = 1000*repeat + fold. Resumable per (tier, repeat) unit:
# each unit writes ckinn_cache/parts/cv_NN_<tier>_r<rep>.csv (about 3 min on the iPad) and is skipped if present.
import pathlib, sys, time
import numpy as np, pandas as pd
if "." not in sys.path:
    sys.path.insert(0, ".")
import ckinn_np as N, ckinn_nn as NN

CACHE = pathlib.Path("ckinn_cache"); PARTS = CACHE / "parts"; PARTS.mkdir(exist_ok=True)
rich = pd.read_csv(CACHE / "rich_harmonized.csv")
FOLDS, REPEATS, TIERS = 5, 3, ['T2', 'T1', 'T3', 'T4']
GRID, TOL, EPOCHS = (0.1, 1.0, 10.0), 0.002, 40            # configs/protocol_v1.json

t_all = time.time()
for tier in TIERS:
    cols = N.TIERS[tier]
    for rep in range(REPEATS):
        out = PARTS / f"cv_NN_{tier}_r{rep}.csv"
        if out.exists():
            print(f"skip {tier} r{rep} (done)"); continue
        t0, rows = time.time(), []
        for k, (tr, te) in enumerate(N.cv_folds(rich, FOLDS, rep)):
            seed = 1000 * rep + k
            for name, make in [('MLP', lambda: NN.DeepCox(cols, lam=0.0, epochs=EPOCHS, seed=seed)),
                               ('CKINN', lambda: NN.CKINNAuto(cols, GRID, TOL, EPOCHS, seed))]:
                fit = N.fit_with_baseline(make(), tr)
                m = N.evaluate(fit, te)
                inner = getattr(fit.model, 'inner_scaled_brier_', {})
                rows.append({'model': name, 'tier': tier, 'repeat': rep, 'fold': k, 'n_train': len(tr), 'n_test': len(te),
                             'lam': getattr(fit.model, 'lam_', 0.0), **m,
                             **{f'inner_sb_{l}': inner.get(l, np.nan) for l in (0.0,) + GRID}})
            print(f"  {tier} r{rep} f{k}: lambda={rows[-1]['lam']}  ({time.time() - t0:.0f} s)")
        pd.DataFrame(rows).to_csv(out, index=False)
        print(f"done {tier} r{rep} in {time.time() - t0:.0f} s")
print(f"total {time.time() - t_all:.0f} s")

parts = sorted(PARTS.glob("cv_NN_*_r*.csv"))
res = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)
res.to_csv(CACHE / "r1_nn.csv", index=False)
show = ['cindex', 'auc_3y', 'scaled_brier', 'oe', 'cal_slope', 'viol_rate']
print(f"\nR1 neural models, {len(parts)} of 12 units done; folds per tier and model (mean, SD):")
print(res.groupby(['tier', 'model'])[show].agg(['mean', 'std']).round(4).to_string())
print("\nlambda chosen by the inner rule, count per tier:")
print(res[res.model == 'CKINN'].groupby('tier').lam.value_counts().unstack(fill_value=0).to_string())
